# 07 - SPECTER Embeddings + Multinomial Logistic Regression

This notebook extracts SPECTER embeddings for `model_text` and trains a multinomial
logistic regression (softmax) classifier on the same splits used in notebook 06.


In [1]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
SPLITS_DIR = Path("data/splits")
EMBEDDINGS_DIR = Path("data/embeddings")
MODEL_DIR = Path("models/specter")
PLOTS_DIR = Path("plots/models")

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


/home/aman/miniconda3/envs/finetuning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv(SPLITS_DIR / "train_papers.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_papers.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_papers.csv")

required_cols = {"model_text", "label"}
for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = required_cols - set(frame.columns)
    if missing:
        raise ValueError(f"{name} split missing columns: {missing}")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["label"])
y_val = label_encoder.transform(val_df["label"])
y_test = label_encoder.transform(test_df["label"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
print("Label order:", list(label_encoder.classes_))


Using device: cpu
Label order: ['APPLIED', 'SURVEY', 'THEORETICAL']


In [3]:
model_candidates = ["allenai/specter2_base", "allenai-specter"]
encoder_model = None
selected_model_name = None

for candidate in model_candidates:
    try:
        print(f"Trying embedding model: {candidate}")
        encoder_model = SentenceTransformer(candidate, device=DEVICE)
        selected_model_name = candidate
        print(f"Loaded model: {selected_model_name}")
        break
    except Exception as exc:
        print(f"Could not load {candidate}: {exc}")

if encoder_model is None:
    raise RuntimeError("Failed to load any SPECTER model candidate.")


def encode_with_cache(split_name: str, texts: pd.Series, batch_size: int = 16) -> np.ndarray:
    model_slug = selected_model_name.replace("/", "_").replace("-", "_")
    cache_path = EMBEDDINGS_DIR / f"specter_{model_slug}_{split_name}.npy"
    if cache_path.exists():
        print(f"Loading cached embeddings: {cache_path}")
        return np.load(cache_path)

    embeddings = encoder_model.encode(
        texts.fillna("").tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    embeddings = embeddings.astype(np.float32)
    np.save(cache_path, embeddings)
    print(f"Saved embeddings: {cache_path}")
    return embeddings


X_train = encode_with_cache("train", train_df["model_text"])
X_val = encode_with_cache("val", val_df["model_text"])
X_test = encode_with_cache("test", test_df["model_text"])

print("Embedding shapes:", X_train.shape, X_val.shape, X_test.shape)


Trying embedding model: allenai/specter2_base


No sentence-transformers model found with name allenai/specter2_base. Creating a new one with mean pooling.


Loaded model: allenai/specter2_base


Batches:   4%|▍         | 5/125 [01:13<29:34, 14.79s/it]


KeyboardInterrupt: 

In [ ]:
candidate_cs = [0.25, 0.5, 1.0, 2.0, 4.0]
val_results = []

for c_val in candidate_cs:
    model = LogisticRegression(
        C=c_val,
        max_iter=5000,
        multi_class="multinomial",
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    macro_f1 = f1_score(y_val, val_pred, average="macro")
    val_results.append({"C": c_val, "val_macro_f1": float(macro_f1)})

val_results_df = pd.DataFrame(val_results).sort_values("val_macro_f1", ascending=False)
best_c = float(val_results_df.iloc[0]["C"])

print("Validation results:")
print(val_results_df)
print(f"Selected C: {best_c}")


In [ ]:
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])

final_model = LogisticRegression(
    C=best_c,
    max_iter=5000,
    multi_class="multinomial",
    solver="lbfgs",
    random_state=RANDOM_STATE,
)
final_model.fit(X_train_val, y_train_val)

y_test_pred = final_model.predict(X_test)

report = classification_report(
    y_test,
    y_test_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    digits=4,
)

metrics = {
    "model": "specter_logreg_multinomial",
    "embedding_model": selected_model_name,
    "best_c": best_c,
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
    "macro_f1": float(f1_score(y_test, y_test_pred, average="macro")),
    "weighted_f1": float(f1_score(y_test, y_test_pred, average="weighted")),
    "labels": list(label_encoder.classes_),
}

report_df = pd.DataFrame(report).transpose()
report_df


In [ ]:
joblib.dump(final_model, MODEL_DIR / "specter_logreg.joblib")
joblib.dump(label_encoder, MODEL_DIR / "label_encoder.joblib")

with open(MODEL_DIR / "specter_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(MODEL_DIR / "specter_model_info.json", "w", encoding="utf-8") as f:
    json.dump({"selected_model": selected_model_name, "device": DEVICE}, f, indent=2)

val_results_df.to_csv(MODEL_DIR / "specter_val_results.csv", index=False)
report_df.to_csv(MODEL_DIR / "specter_classification_report.csv", index=True)

cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("SPECTER + Multinomial Logistic Regression")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "specter_confusion_matrix.png", dpi=160)
plt.close(fig)

comparison_rows = [{"model": "SPECTER+LogReg", **metrics}]
tfidf_metrics_path = Path("models/tfidf/tfidf_metrics.json")
if tfidf_metrics_path.exists():
    with open(tfidf_metrics_path, "r", encoding="utf-8") as f:
        tfidf_metrics = json.load(f)
    comparison_rows.append(
        {
            "model": "TFIDF+LogReg",
            "accuracy": tfidf_metrics.get("accuracy"),
            "macro_f1": tfidf_metrics.get("macro_f1"),
            "weighted_f1": tfidf_metrics.get("weighted_f1"),
            "best_c": tfidf_metrics.get("best_c"),
            "embedding_model": "tfidf",
            "labels": tfidf_metrics.get("labels"),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(Path("models/model_comparison.csv"), index=False)

print("Saved SPECTER artifacts to models/specter and plots/models.")
print("Saved model comparison to models/model_comparison.csv")
comparison_df
